# Import Libabrys

In [3]:
import pandas as pd
import os

# Mount Google Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Define directory paths

In [9]:
folder_path = '/content/drive/MyDrive/FYP Data Cleaning/Raw Data/'
output_dir  = '/content/drive/MyDrive/FYP Data Cleaning/Clean Data/'

file_name = 'top.json'
file_path = os.path.join(folder_path, file_name)

# Load the file in chunks

In [11]:
chunk_size = 50000
chunks = []

print(f"Reading file in chunks: {file_name}")
for chunk in pd.read_json(file_path, lines=True, chunksize=chunk_size):
    chunks.append(chunk)

df_json = pd.concat(chunks, ignore_index=True)
del chunks

print("\n--- Initial Shape ---")
print(df_json.shape)

Reading file in chunks: top.json

--- Initial Shape ---
(250000, 9)


# Display Missing values

In [12]:
print(df_json.isnull().sum())

id                          0
title                       0
selftext                    0
z_score                     0
normalized_score            0
subreddit                   0
body                        0
comment_normalized_score    0
combined_score              0
dtype: int64


# Count of duplicates

In [13]:
dup_full_row = df_json.duplicated().sum()
dup_by_id    = df_json.duplicated(subset=['id']).sum()
dup_id_body  = df_json.duplicated(subset=['id', 'body']).sum()

print("--- Duplicate Check (before cleaning) ---")
print(f"Total rows                          : {len(df_json)}")
print(f"Duplicate rows (all columns)        : {dup_full_row}")
print(f"Duplicate rows (by id only)         : {dup_by_id}")
print(f"Duplicate rows (by id + body)       : {dup_id_body}")
print(f"Unique ids                          : {df_json['id'].nunique()}")

--- Duplicate Check (before cleaning) ---
Total rows                          : 250000
Duplicate rows (all columns)        : 7
Duplicate rows (by id only)         : 131679
Duplicate rows (by id + body)       : 34
Unique ids                          : 118321


# Remove duplicates

In [14]:
before = len(df_json)
df_json = df_json.drop_duplicates().reset_index(drop=True)

print(f"Removed   : {before - len(df_json)}")
print(f"Remaining : {len(df_json)}")

Removed   : 7
Remaining : 249993


# Verify

In [16]:
print("--- Cleaned Shape ---")
print(df_json.shape)

print("\n--- Remaining Missing Values ---")
print(df_json.isnull().sum())

print("\n--- Sample ---")
print(df_json.head(2))

--- Cleaned Shape ---
(249993, 9)

--- Remaining Missing Values ---
id                          0
title                       0
selftext                    0
z_score                     0
normalized_score            0
subreddit                   0
body                        0
comment_normalized_score    0
combined_score              0
dtype: int64

--- Sample ---
       id                                              title  \
0   utf5u         Where has all the money in the world gone?   
1  m3g13g  Is there a better sub where comments aren’t hi...   

                                            selftext    z_score  \
0  Honest question.\n\nWhere is all the money?  I...  34.131260   
1  So often someone will ask an amazing question,...  14.812156   

   normalized_score     subreddit  \
0               1.0       finance   
1               1.0  AskEconomics   

                                                body  \
0  (relix already hit on some of this)\n\nIt's ha...   
1  I someone w

# Save the cleaned dataset

In [17]:
output_path = os.path.join(output_dir, 'reddit_finance_43_250K_Clean_dataset.csv')
df_json.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")

Saved to: /content/drive/MyDrive/FYP Data Cleaning/Clean Data/reddit_finance_43_250K_Clean_dataset.csv
